In [1]:
import json
import os
from os.path import join
import requests
import re
from nltk.tokenize import word_tokenize,sent_tokenize
from nltk import pos_tag
from tqdm import tqdm
from nltk.corpus import wordnet as wn
from collections import Counter

import warnings
warnings.filterwarnings('ignore')

from nltk.corpus import stopwords
stopword_list = stopwords.words('english')
stopword_list.extend(['``','\'s','\'\'','n\'t'])

def wordnet_hypernyms(token):
    all_hypernyms = []

    word_senses = wn.synsets(token)
    hypernyms = lambda s: s.hypernyms()
    for ws in word_senses:
        hypernyms = [hyp.name() for hyp in list(ws.closure(hypernyms))]
        all_hypernyms.extend(hypernyms)
    return all_hypernyms

def intersection(list1,list2):
    return list(set(list1) & set(list2))
    

def remove_punctuation_and_stopwords(words):
    new_list= []
    for w in words:
        if w.isalnum() and w not in stopword_list:
            new_list.append( w )
    return new_list

In [2]:
json_data = json.load(open('reviews.json'))


In [3]:
dir = 'Lexicon'
if not os.path.isdir(dir):
    os.mkdir(dir)
    
base_url = 'https://raw.githubusercontent.com/peterverhaar/dark_academia/refs/heads/main/Lexicon/'

lexicon_files = [
    'academia.txt',
    'literature_and_culture.txt',
    'mood.txt',
    'objects.txt',
    'spaces.txt'
]
    

for l in lexicon_files:
    topic = l[ : l.rindex('.') ]
    response = requests.get( base_url + l)
    words = []
    if response:
        response.encoding = 'utf-8'
        out = open( os.path.join( dir , l ) , 'w' , encoding = 'utf-8' )
        out.write( response.text )
        out.close()

print('Lexicons have been downloaded!')



lexicons = dict()


for file in os.listdir(dir):
    if re.search(r'txt$',file):
    
        topic = re.sub( r'\.txt$','',file )
        words = []

        with open( join(dir,file) , encoding = 'utf-8' ) as file_handler:   
            for l in file_handler: 
                if re.search( r'\w' , l ):
                    words.append(l.strip().lower())

        lexicons[topic] = words    


Lexicons have been downloaded!


In [4]:
combined = dict()

for trope in lexicons:
    patterns = lexicons[trope]
    combined[trope] = "(" + ")|(".join(patterns) + ")"
    
    

In [ ]:
all_data = []

for review in tqdm(json_data):
    row = dict()
    review_text = review['text']
    row['id'] = review['review_id']

    annotations = []


    sentences = sent_tokenize(review_text)
    for sentence in sentences:
        
        for trope in combined:
            if re.search(rf'{combined[trope]}',sentence):
                annotation_dict = dict()
                annotation_dict['sentence'] = sentence
                annotation_dict[trope] = True
                annotations.append(annotation_dict)

 
    if len(annotations)>0:
        row['annotations'] = annotations
    
    all_data.append(row)     

 24%|█████████▎                            | 2506/10239 [00:39<02:28, 52.13it/s]

In [ ]:
with open('annotations.json','w',encoding='utf-8') as out:
    out.write( json.dumps(all_data, indent=4) )

In [ ]:
review_base_url = 'https://www.goodreads.com/review/show/'

out = open('annotations.tsv','w',encoding='utf-8')
out.write('sentence\tclaim\treview_type\treview\n')
          

for row in all_data:

    review_id = row['id']
    review_type = row['type']
    if 'annotations' in row:
        for annotation in row['annotations']:
            #print(annotation)
            sentence = annotation['sentence']
            claim = ''
            if 'reference_to_plot' in annotation:
                claim = 'reference_to_plot'
            elif 'text_difficulty' in annotation:
                claim = 'text_difficulty'
            elif 'emotion_type' in annotation:
                claim = f'Emotion_{annotation["emotion_type"]}'
            
            out.write(f'{sentence}\t{claim}\t{review_type}\t{review_base_url}{review_id}\n')

            
out.close()            
    